# Deploying AG2 Agents as A2A Microservices with Bindu

This Jupyter Notebook demonstrates how to take an AG2 (formerly AutoGen) conversational workflow and expose it as a production-ready microservice using [Bindu](https://getbindu.com/).

While AG2 excels at orchestrating complex multi-agent logic, Bindu provides the networking and protocol layer. By wrapping your AG2 agents in Bindu, they instantly gain:
1. **A2A Protocol Support:** Standardized JSON-RPC communication.
2. **Decentralized Identity (DIDs):** Cryptographic signatures for secure interactions.
3. **State Management:** Built-in task lifecycle tracking.

## Requirements

AG2 requires `Python>=3.10`. To run this notebook example, please install:
````{=mdx}
:::info Requirements
Install `ag2` and `bindu`:
```bash
pip install "ag2[openai]" bindu
```

For more information on Bindu's architecture, refer to the [Bindu GitHub repository](https://github.com/getbindu/Bindu).
:::
````

In [ ]:
%%capture --no-stderr
# %pip install "ag2[openai]>=0.11.0" bindu

## Defining the AG2 Agent and Bindu Handler

To deploy an AG2 agent as a Bindu microservice, we need to define a `handler` function. This function intercepts incoming requests from the A2A network, initializes our AG2 agent to maintain a stateless API environment, and returns the generated response.

Because starting a web server blocks the Jupyter Notebook execution, we will use the `%%writefile` magic command to save our microservice code to a Python script.

In [ ]:
%%writefile ag2_bindu_server.py
import logging
import os

from autogen import ConversableAgent, LLMConfig
from bindu.penguin import bindufy

logger = logging.getLogger(__name__)
logger.setLevel(logging.WARNING)

# 1. Configure the LLM for AG2
llm_config = LLMConfig(
    config_list=[
        {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY", ""),
        }
    ]
)

# 2. Define the Bindu configuration
config = {
    "author": "ag2.developer@example.com",
    "name": "ag2-networked-assistant",
    "description": "An AG2 ConversableAgent exposed via the Bindu A2A protocol.",
    "deployment": {
        "url": "http://localhost:3773",
        "expose": True, 
    },
    "skills": []
}

# 3. Create the Handler to bridge Bindu and AG2
def handler(messages: list[dict[str, str]]):
    """
    Intercepts standard A2A protocol messages, passes them to AG2, 
    and returns the structured response.
    """
    if not messages:
        return [{"role": "assistant", "content": "No input provided."}]
    
    user_input = messages[-1].get("content", "")
    if not user_input:
        return [{"role": "assistant", "content": "Empty message."}]
    
    # Initialize fresh agents per request to maintain stateless API behavior
    assistant = ConversableAgent(
        name="assistant",
        system_message="You are a helpful AG2 assistant. Keep answers concise.",
        llm_config=llm_config,
    )
    
    user_proxy = ConversableAgent(
        name="user_proxy", 
        human_input_mode="NEVER",
    )
    
    # Run the AG2 conversation
    result = user_proxy.initiate_chat(
        assistant, 
        message=user_input, 
        max_turns=1
    )
    
    # Extract and return the final response
    if result.chat_history:
        reply = result.chat_history[-1].get("content", "")
        return [{"role": "assistant", "content": reply}]
        
    return [{"role": "assistant", "content": "Task completed."}]

if __name__ == "__main__":
    # Launch the AG2 agent as a Bindu microservice
    bindufy(config, handler)


## Running Your Microservice

````{=mdx}
:::tip
You can run the generated Python script directly from your terminal. Ensure your OpenAI API key is set in your environment variables.
:::
````

To start the Bindu A2A server, open your terminal and run:

```bash
export OPENAI_API_KEY="sk-your-key-here"
python ag2_bindu_server.py
```

Your AG2 agent is now live at `http://localhost:3773`. It is fully compliant with the A2A protocol and ready to securely interact with other agents or frontend interfaces on the Bindu network!